# Additional sparse-surrogate families

This companion audits cached outcomes from the fixed families_v2 protocol.
It does not describe a cache audit as a new GPU inference run. Source
modules perform extraction and evaluation; their hashes are recorded in
all manifests. The same documents as external_v1 are reused.

Conditions: Pythia TopK, SmolLM2 MLP-output TopK, Qwen ReLU transcoder,
and Gemma Matryoshka with full, 512-, and 2,048-coordinate views. Prefixes
are not independently trained checkpoints. Llama and DeepSeek candidate
exclusions are documented separately, not counted as evaluated models.


In [1]:
from pathlib import Path
import json
import sys

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/lattmc/activationstudy").is_dir())
sys.path.insert(0, str(root / "src"))
output = root / "data/activation_studies/families_v2"
protocol = root / "experiments/activation_studies/families_v2/PROTOCOL.md"
print("Cache:", output)


Cache: /Users/ltsinadze/git/my_papers/data/activation_studies/families_v2


## Independent score and provenance audit

Recompute AP and thresholded confusion counts from saved score arrays,
check split isolation and checksums, and verify prefix arrays are exactly
the corresponding columns of the full dictionary. No raw text is needed.


In [2]:
from lattmc.activationstudy.families_audit_codexgen import audit

result = audit(output, protocol)
print(json.dumps(result, indent=2))


{
  "query_configurations": 4320,
  "metric_vectors": 56160,
  "test_score_comparisons": 35568000,
  "status": "passed",
  "exact_prefix_checks": 8,
  "protocol_sha256": "2d69a794b2c92103fa8afd2ecaf15c21d9591a251b25378fe3fbd4ed75790e43"
}


In [3]:
from lattmc.activationstudy.families_config_codexgen import CHECKPOINTS

for dataset in ("ag_news", "dbpedia_14"):
    print("\n", dataset, "maximum pooling, three positive sources")
    for name in CHECKPOINTS:
        path = output / dataset / f"{name}_max_results.json"
        summary = json.loads(path.read_text())["summary"]["3"]
        means = summary["methods"]
        print(name, {method: round(100 * means[method]["mean"], 2)
                     for method in ("graded", "support", "best_single",
                                    "sae_probe", "dense_probe")})



 ag_news maximum pooling, three positive sources
pythia_topk {'graded': 29.34, 'support': 28.01, 'best_single': 65.69, 'sae_probe': 41.74, 'dense_probe': 56.63}
smol_topk {'graded': 27.16, 'support': 26.61, 'best_single': 51.31, 'sae_probe': 57.04, 'dense_probe': 58.22}
qwen_transcoder {'graded': 31.97, 'support': 30.44, 'best_single': 37.29, 'sae_probe': 41.57, 'dense_probe': 62.0}
gemma_matryoshka {'graded': 28.71, 'support': 27.64, 'best_single': 62.2, 'sae_probe': 61.37, 'dense_probe': 73.35}
gemma_matryoshka_512 {'graded': 32.28, 'support': 27.88, 'best_single': 56.16, 'sae_probe': 44.67, 'dense_probe': 73.35}
gemma_matryoshka_2048 {'graded': 34.92, 'support': 29.89, 'best_single': 66.74, 'sae_probe': 54.53, 'dense_probe': 73.35}

 dbpedia_14 maximum pooling, three positive sources
pythia_topk {'graded': 40.14, 'support': 26.69, 'best_single': 71.74, 'sae_probe': 42.25, 'dense_probe': 46.83}
smol_topk {'graded': 22.56, 'support': 19.32, 'best_single': 69.01, 'sae_probe': 61.11, '

In [4]:
for dataset in ("ag_news", "dbpedia_14"):
    print("\n", dataset, "held-out token diagnostics")
    for name in CHECKPOINTS:
        path = output / dataset / f"{name}_extraction.json"
        stats = json.loads(path.read_text())["statistics"]
        print(name, "L0", round(stats["mean_token_l0"], 2),
              "uncentered NMSE", round(stats["uncentered_nmse"], 4))



 ag_news held-out token diagnostics
pythia_topk L0 20.0 uncentered NMSE 0.0286
smol_topk L0 32.0 uncentered NMSE 0.2191
qwen_transcoder L0 10.41 uncentered NMSE 0.5109
gemma_matryoshka L0 96.31 uncentered NMSE 0.1606
gemma_matryoshka_512 L0 26.47 uncentered NMSE 0.3846
gemma_matryoshka_2048 L0 43.76 uncentered NMSE 0.3014

 dbpedia_14 held-out token diagnostics
pythia_topk L0 20.0 uncentered NMSE 0.0343
smol_topk L0 32.0 uncentered NMSE 0.189
qwen_transcoder L0 10.67 uncentered NMSE 0.4565
gemma_matryoshka L0 90.16 uncentered NMSE 0.1534
gemma_matryoshka_512 L0 26.2 uncentered NMSE 0.3757
gemma_matryoshka_2048 L0 42.22 uncentered NMSE 0.292


## Re-extraction and evaluation

Run these explicit commands from the repository root with its existing
uv environment. They require the pinned upstream models and local token
arrays. Use the protocol README to reconstruct missing inputs. Accelerator
inference was executed by these source modules, outside this notebook.

```sh
uv run python -m lattmc.activationstudy.families_extract_codexgen \
  --output data/activation_studies/families_v2 \
  --family pythia_topk
```

Repeat for `smol_topk`, `qwen_transcoder`, and `gemma_matryoshka`, then:

```sh
uv run python -m lattmc.activationstudy.families_evaluate_codexgen \
  --output data/activation_studies/families_v2 \
  --protocol experiments/activation_studies/families_v2/PROTOCOL.md
```

Interpret intervals as conditional source-draw variability. These fixed
English corpora, reused categories, and heterogeneous checkpoints do not
establish causal architecture effects or semantic feature coherence.


## Query type and token-witness presentation

This phase uses pooled document meets,
not meets of selected title/name tokens. Maximum pooling implements the
coordinatewise join; mean pooling is a separate baseline. The complete
fixed 24-row maximum-pooling gallery is inspected in
`context_depth_codexgen.ipynb` and rendered by
`lattmc.contextstudy.witnessrender_codexgen`. No matched external
SAE/transcoder pair or named-token meet/join sweep is implied.